In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import cv2
import numpy as np

# Input folders
input_dirs = {
    "benign": "/content/drive/MyDrive/DDSM/DDSM_unzipped/Benign",
    #"cancer": "/content/drive/MyDrive/DDSM/DDSM_unzipped/Cancer"
}

# Output folders
output_dirs = {
    "benign": "/content/drive/MyDrive/DDSM/ROI_Benign",
    #"cancer": "/content/drive/MyDrive/DDSM/ROI_Cancer"
}

# Create output folders
for d in output_dirs.values():
    os.makedirs(d, exist_ok=True)

def crop_roi(image, mask, size=224, padding=0.1):
    """Crop the ROI from the mask and resize it to a square."""
    coords = cv2.findNonZero(mask)
    if coords is None:  # No lesion found
        return None
    x, y, w, h = cv2.boundingRect(coords)

    # Expand to a square around the center
    max_side = max(w, h)
    x_center, y_center = x + w // 2, y + h // 2
    half_side = int(max_side / 2 * (1 + padding))

    x1 = max(0, x_center - half_side)
    y1 = max(0, y_center - half_side)
    x2 = min(image.shape[1], x_center + half_side)
    y2 = min(image.shape[0], y_center + half_side)

    roi = image[y1:y2, x1:x2]
    roi_resized = cv2.resize(roi, (size, size))
    return roi_resized

# Initialize statistics
stats = {
    "benign": {"roi_count": 0, "no_mask": 0},
    #"cancer": {"roi_count": 0, "no_mask": 0}
}

# Iterate over benign / cancer folders
for label, in_dir in input_dirs.items():
    out_dir = output_dirs[label]

    case_list = os.listdir(in_dir)
    total_cases = len(case_list)
    case_counter = 0

    # Iterate over each case folder
    for case in os.listdir(in_dir):
        case_path = os.path.join(in_dir, case)
        if not os.path.isdir(case_path):
            continue
        case_counter += 1
        print(f"Processing class {label}, case {case_counter}/{total_cases}: {case}")

        for fname in os.listdir(case_path):
            if fname.endswith(".jpg") and "Mask" not in fname:
                image_path = os.path.join(case_path, fname)
                mask_path = os.path.join(case_path, fname.replace(".jpg", "_Mask.jpg"))


                # If there is no mask, count it but do not resize or save
                if not os.path.exists(mask_path):
                    stats[label]["no_mask"] += 1
                    continue  # Skip this image

                # Read the image and mask
                image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
                mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

                # Binarize the mask
                _, mask_bin = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

                roi = crop_roi(image, mask_bin)
                if roi is None:
                    stats[label]["no_mask"] += 1
                    continue  # Skip if the mask is empty or no lesion is present

                # Apply CLAHE to the cropped ROI
                clahe = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))
                clahe_roi = clahe.apply(roi)

                # Save to the output folder
                out_name = f"{case}_{fname}"
                out_path = os.path.join(out_dir, out_name)
                cv2.imwrite(out_path, clahe_roi)

                stats[label]["roi_count"] += 1

# Print summary statistics
print("ROI cropping is complete. Summary:")
for label in stats:
    print(f"{label.capitalize()} Class: ROI  = {stats[label]['roi_count']}, images without mask = {stats[label]['no_mask']}")


正在处理 benign 类别，第 1/671 个病例: 0029
正在处理 benign 类别，第 2/671 个病例: 0033
正在处理 benign 类别，第 3/671 个病例: 0217
正在处理 benign 类别，第 4/671 个病例: 0234
正在处理 benign 类别，第 5/671 个病例: 0235
正在处理 benign 类别，第 6/671 个病例: 0236
正在处理 benign 类别，第 7/671 个病例: 0237
正在处理 benign 类别，第 8/671 个病例: 0238
正在处理 benign 类别，第 9/671 个病例: 0239
正在处理 benign 类别，第 10/671 个病例: 0240
正在处理 benign 类别，第 11/671 个病例: 0241
正在处理 benign 类别，第 12/671 个病例: 0242
正在处理 benign 类别，第 13/671 个病例: 0243
正在处理 benign 类别，第 14/671 个病例: 0245
正在处理 benign 类别，第 15/671 个病例: 0246
正在处理 benign 类别，第 16/671 个病例: 0247
正在处理 benign 类别，第 17/671 个病例: 0248
正在处理 benign 类别，第 18/671 个病例: 0249
正在处理 benign 类别，第 19/671 个病例: 0250
正在处理 benign 类别，第 20/671 个病例: 0251
正在处理 benign 类别，第 21/671 个病例: 0252
正在处理 benign 类别，第 22/671 个病例: 0253
正在处理 benign 类别，第 23/671 个病例: 0254
正在处理 benign 类别，第 24/671 个病例: 0255
正在处理 benign 类别，第 25/671 个病例: 0256
正在处理 benign 类别，第 26/671 个病例: 0257
正在处理 benign 类别，第 27/671 个病例: 0258
正在处理 benign 类别，第 28/671 个病例: 0259
正在处理 benign 类别，第 29/671 个病例: 0270
正在处理 benign 类别，第 30/671

In [ ]:
import os
import cv2
import numpy as np

# Input folders
input_dirs = {
    #"benign": "/content/drive/MyDrive/DDSM/DDSM_unzipped/Benign",
    "cancer": "/content/drive/MyDrive/DDSM/DDSM_unzipped/Cancer"
}

# Output folders
output_dirs = {
    #"benign": "/content/drive/MyDrive/DDSM/ROI_Benign",
    "cancer": "/content/drive/MyDrive/DDSM/ROI_Cancer"
}

# Create output folders
for d in output_dirs.values():
    os.makedirs(d, exist_ok=True)

def crop_roi(image, mask, size=224, padding=0.1):
    """Crop the ROI from the mask and resize it to a square."""
    coords = cv2.findNonZero(mask)
    if coords is None:  # No lesion found
        return None
    x, y, w, h = cv2.boundingRect(coords)

    # Expand to a square around the center
    max_side = max(w, h)
    x_center, y_center = x + w // 2, y + h // 2
    half_side = int(max_side / 2 * (1 + padding))

    x1 = max(0, x_center - half_side)
    y1 = max(0, y_center - half_side)
    x2 = min(image.shape[1], x_center + half_side)
    y2 = min(image.shape[0], y_center + half_side)

    roi = image[y1:y2, x1:x2]
    roi_resized = cv2.resize(roi, (size, size))
    return roi_resized

# Initialize statistics
stats = {
    #"benign": {"roi_count": 0, "no_mask": 0},
    "cancer": {"roi_count": 0, "no_mask": 0}
}

# Iterate over benign / cancer folders
for label, in_dir in input_dirs.items():
    out_dir = output_dirs[label]

    case_list = os.listdir(in_dir)
    total_cases = len(case_list)
    case_counter = 0

    # Iterate over each case folder
    for case in os.listdir(in_dir):
        case_path = os.path.join(in_dir, case)
        if not os.path.isdir(case_path):
            continue
        case_counter += 1

        for fname in os.listdir(case_path):
            if fname.endswith(".jpg") and "Mask" not in fname:
                image_path = os.path.join(case_path, fname)
                mask_path = os.path.join(case_path, fname.replace(".jpg", "_Mask.jpg"))


                # If there is no mask, count it but do not resize or save
                if not os.path.exists(mask_path):
                    stats[label]["no_mask"] += 1
                    continue  # Skip this image

                # Read the image and mask
                image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
                mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

                # Binarize the mask
                _, mask_bin = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

                roi = crop_roi(image, mask_bin)
                if roi is None:
                    stats[label]["no_mask"] += 1
                    continue  # Skip if the mask is empty or no lesion is present

                # Apply CLAHE to the cropped ROI
                clahe = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))
                clahe_roi = clahe.apply(roi)

                # Save to the output folder
                out_name = f"{case}_{fname}"
                out_path = os.path.join(out_dir, out_name)
                cv2.imwrite(out_path, clahe_roi)

                stats[label]["roi_count"] += 1

# Print summary statistics
print("ROI cropping is complete. Summary:")
for label in stats:
    print(f"{label.capitalize()} Class: ROI  = {stats[label]['roi_count']}, images without mask = {stats[label]['no_mask']}")

ROI cropping is complete. Summary:
Cancer Class: ROI  = 1428, images without mask = 1288
